# 04 - Severity Model Development

Develop and validate the severity scoring model.
Currently rule-based; will be upgraded to ML with sufficient training data.

In [ ]:
import sys
sys.path.insert(0, '../../backend')

from app.ml.severity_model import SeverityModel, SeverityInput
import matplotlib.pyplot as plt
import numpy as np

model = SeverityModel()

In [ ]:
# Test scenarios
scenarios = [
    ('Fender bender', SeverityInput(crash_probability=0.3, vehicle_count=2, vehicle_types=['car', 'car'],
        optical_flow_magnitude=1.0, motion_anomaly_score=0.2, lane_impact='right_shoulder', hour_of_day=14)),
    ('Multi-vehicle', SeverityInput(crash_probability=0.7, vehicle_count=4, vehicle_types=['car', 'truck', 'car', 'car'],
        optical_flow_magnitude=3.0, motion_anomaly_score=0.6, lane_impact='lanes_1_2', hour_of_day=17)),
    ('Major pileup', SeverityInput(crash_probability=0.95, vehicle_count=8, vehicle_types=['car']*6 + ['truck']*2,
        optical_flow_magnitude=5.0, motion_anomaly_score=0.9, lane_impact='all_lanes', hour_of_day=8, has_fire=True)),
]

for name, features in scenarios:
    result = model.predict(features)
    print(f'{name}: score={result.score:.3f}, level={result.level}')
    print(f'  Factors: {result.factors}\n')

In [ ]:
# Visualize severity distribution across crash probabilities
probs = np.linspace(0, 1, 100)
scores = []
for p in probs:
    result = model.predict(SeverityInput(
        crash_probability=p, vehicle_count=2, vehicle_types=['car', 'car'],
        optical_flow_magnitude=2.0, motion_anomaly_score=p*0.5,
        lane_impact='lane_1', hour_of_day=12
    ))
    scores.append(result.score)

plt.figure(figsize=(10, 5))
plt.plot(probs, scores)
plt.axhline(y=0.25, color='blue', linestyle='--', alpha=0.5, label='minor/moderate')
plt.axhline(y=0.50, color='orange', linestyle='--', alpha=0.5, label='moderate/severe')
plt.axhline(y=0.75, color='red', linestyle='--', alpha=0.5, label='severe/critical')
plt.xlabel('Crash Probability')
plt.ylabel('Severity Score')
plt.title('Severity Score vs Crash Probability')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()